# IS 445 HW5 — UFO Sightings Visualizations
**Dataset:** [UFO Sightings (scrubbed, geocoded, time-standardized)](https://github.com/UIUC-iSchool-DataViz/is445_data/raw/main/ufo-scrubbed-geocoded-time-standardized-00.csv)

In [9]:
import sys
!{sys.executable} -m pip install altair vega_datasets

In [10]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

## Load & Clean Data

In [11]:
url = "https://github.com/UIUC-iSchool-DataViz/is445_data/raw/main/ufo-scrubbed-geocoded-time-standardized-00.csv"

df = pd.read_csv(url)
df.columns = [
    'datetime', 'city', 'state', 'country', 'shape',
    'duration_seconds', 'duration_hours_min', 'comments',
    'date_posted', 'latitude', 'longitude'
]

# Keep US sightings 
df = df[df['country'] == 'us'].copy()

# Coerce numeric columns
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude']= pd.to_numeric(df['longitude'], errors='coerce')
df['duration_seconds'] = pd.to_numeric(df['duration_seconds'], errors='coerce')

# Drop rows missing key
df = df.dropna(subset=['latitude', 'longitude', 'shape'])

# Filter to continental US 
df = df[
    (df['latitude']  >  24) & (df['latitude']  <  50) &
    (df['longitude'] > -125) & (df['longitude'] < -66)
].copy()

# Parse datetime and extract year
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
df['year'] = df['datetime'].dt.year
df = df.dropna(subset=['year'])
df['year'] = df['year'].astype(int)

print(f"Rows after cleaning: {len(df):,}")
df.head(3)

Rows after cleaning: 62,968


,datetime,city,state,country,shape,duration_seconds,duration_hours_min,comments,date_posted,latitude,longitude,year
2,1956-10-10 21:00:00,edna,tx,us,circle,20.0,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833,1956
4,1961-10-10 19:00:00,bristol,tn,us,sphere,300.0,5 minutes,My father is now 89 my brother 52 the girl wit...,4/27/2007,36.595000,-82.188889,1961
6,1965-10-10 23:45:00,norwalk,ct,us,disk,1200.0,20 minutes,A bright orange color changing to reddish colo...,10/2/1999,41.117500,-73.408333,1965


## Plot 1 — Interactive Geographic Scatter (hover to highlight by shape)

I restrict to the **top 10 most-reported UFO shapes** and sample 6,000 rows to keep the vega-lite spec size reasonable.

In [12]:
# Keep only the 10 most common shapes
top_shapes = df['shape'].value_counts().head(10).index.tolist()
df_top = df[df['shape'].isin(top_shapes)].copy()

# Sample for performance
df_sample = df_top.sample(6000, random_state=42)

# Interactivity: hover selection highlights one shape at a time 
hover = alt.selection_point(
    on='mouseover',
    fields=['shape'],
    empty=False          # when nothing is hovered, show all at full opacity
)

color_enc = alt.Color(
    'shape:N',
    scale=alt.Scale(scheme='tableau10'),
    legend=alt.Legend(title='UFO Shape')
)
#: :generated:
#: :model: Claude Sonnet 4.6
#: :prompt: Using my given code, create an Altair scatter plot of longitude vs latitude for continental US sightings. Add interactivity using selection_point on mouseover bound to the shape field, so hovering highlights all points of the same shape. Include tooltips for city, state, shape, year, and duration_seconds.
#: :response:
# Create Altair scatter plot with interactivity
chart1 = (
    alt.Chart(df_sample)
    .mark_circle()
    .encode(
        x=alt.X('longitude:Q',
                scale=alt.Scale(domain=[-125, -66]),
                title='Longitude'),
        y=alt.Y('latitude:Q',
                scale=alt.Scale(domain=[24, 50]),
                title='Latitude'),
        color=alt.condition(hover, color_enc, alt.value('lightgray')),
        size=alt.condition(hover, alt.value(55), alt.value(12)),
        opacity=alt.condition(hover, alt.value(0.85), alt.value(0.25)),
        tooltip=[
            alt.Tooltip('city:N', title='City'),
            alt.Tooltip('state:N', title='State'),
            alt.Tooltip('shape:N', title='Shape'),
            alt.Tooltip('year:Q', title='Year'),
            alt.Tooltip('duration_seconds:Q', title='Duration (s)', format='.0f'),
        ]
    )
    .add_params(hover)
    .properties(
        width=620,
        height=380,
        title='UFO Sightings Across the Continental United States (Top 10 Shapes)'
    )
)

chart1
#: :end:

alt.Chart(...)

In [13]:
import os
chart1.save(os.path.expanduser('~/ufo_map.json'))

## Plot 2 — Bar Chart: Sightings per Year (1950–2014)

In [14]:
#: :generated:
#: :model: Claude Sonnet 4.6
#: :prompt: Using the cleaned UFO dataframe, aggregate sightings by year for 1950-2014 and create an Altair bar chart. Encode year as ordinal on x, count as quantitative on y, and also encode count with a sequential blues color scheme to show magnitude. Label x-axis every 5 years at a -45 degree angle. Include tooltips.
#: :response:
# Aggregate by year
df_year = (
    df[(df['year'] >= 1950) & (df['year'] <= 2014)]
    .groupby('year')
    .size()
    .reset_index(name='count')
)

chart2 = (
    alt.Chart(df_year)
    .mark_bar()
    .encode(
        x=alt.X('year:O',
                title='Year',
                axis=alt.Axis(
                    labelAngle=-45,
                    values=list(range(1950, 2015, 5))
                )),
        y=alt.Y('count:Q', title='Number of Sightings'),
        color=alt.Color(
            'count:Q',
            scale=alt.Scale(scheme='blues'),
            legend=alt.Legend(title='Sighting Count')
        ),
        tooltip=[
            alt.Tooltip('year:O',  title='Year'),
            alt.Tooltip('count:Q', title='Sightings')
        ]
    )
    .properties(
        width=620,
        height=300,
        title='UFO Sightings Reported per Year in the US (1950–2014)'
    )
)

chart2
#: :end:

alt.Chart(...)

In [15]:
chart2.save(os.path.expanduser('~/ufo_yearly.json'))

**Writing with Claude's help.**
